In [1]:
import pandas as pd
import json


In [2]:
MODELS = [
    "gpt-4o",
    "gpt-3.5-turbo",
    "meta-llama_Llama-3-70b-chat-hf",
    "mistralai_Mixtral-8x22B-Instruct-v0.1",
    "o3-mini"
]

In [3]:
THEORIES = [
    "dyadic"
]

In [4]:
model="gpt-4o"
theory="dyadic"
end_to_end_result = pd.read_csv(f"../results/{model}/end_to_end.tsv", sep="\t")
theory_result = pd.read_csv(f"../results/{model}/dyadic.tsv", sep="\t")
theory_result = theory_result.iloc[146:]

In [14]:
theory_result.groupby(["dataset", "id"]).first().reset_index()

,dataset,id,scenario,circumstance_type,circumstance,vulnerable_score,intentional_score,harm_score,help_score,moral_score,vulnerable_expl,intentional_expl,harm_expl,help_expl,moral_expl,agent,assumption_for_agent,patient,assumption_for_patient,human_score
0,Clifford,155,You see a girl asking for allowance even thoug...,orig,NaN,6.0,8.0,4.0,0.0,-2.0,The brother is of average resilience.,The girl intentionally asked for allowance.,"The harm is minor, mostly unfairness.",No help was provided.,The action is somewhat immoral due to unfairness.,The girl,She is asking for allowance.,The brother,He did the chores.,-2.70
1,Clifford,156,You see a soccer player pretending to be serio...,orig,NaN,6.0,8.0,5.0,0.0,-3.0,The opposing player is an adult athlete,The action is premeditated to deceive,The harm is reputational and could affect the ...,No help is provided by the action,Deception in sports is generally considered im...,The soccer player pretending to be fouled,The player is trying to deceive the referee,The opposing player,The player is being falsely accused,-2.50
2,Clifford,160,You see someone at the school gym lifting free...,orig,NaN,0.0,8.0,0.0,0.0,0.0,Free weights are inanimate and cannot be harmed,Lifting weights is a deliberate action,No harm is caused by lifting weights,No help is provided by lifting weights,Neutral action with no moral implications,A person in business clothes,They might be a professional who came directly...,The free weights,They are inanimate objects,-0.20
3,Clifford,161,You see a man secretly voting against his wife...,orig,NaN,7.0,8.0,5.0,0.0,-3.0,She is vulnerable due to emotional stakes.,The action was deliberate and planned.,The harm is emotional and reputational.,No help was provided.,The action is deceitful and harmful.,The man,He is the husband of the patient.,The wife,She is participating in a beauty pageant.,-1.90
4,Clifford,164,You see a homosexual in a gay bar offering sex...,orig,NaN,6.0,8.0,5.0,2.0,0.0,The patient is an adult with autonomy,The action is deliberate and planned,Potential for emotional or social harm,No significant help provided,Morally neutral with potential risks,The homosexual offering sex,The person is seeking attention or validation,Anyone who buys him a drink,The person is willing to engage in the offer,-2.60
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
498,Mickelberg,633,Person X learned a secret prize-winning pie re...,orig,NaN,6.0,8.0,0.0,7.0,3.0,Person X is an average adult.,Learning a recipe involves deliberate effort.,No harm was caused by learning the recipe.,Gaining a valuable recipe is substantial help.,Learning a recipe is morally neutral to positive.,Person X,Person X is the one who learned the recipe.,Person X,Person X is affected by gaining knowledge.,0.48
499,Mickelberg,635,Person X was singing loudly to their favorite ...,orig,NaN,0.0,8.0,0.0,0.0,0.0,No one is directly affected.,Singing is a deliberate action.,No harm caused by singing.,No help provided by singing.,Neutral action with no moral implications.,Person X,Person X is an adult.,No specific patient,No one is directly affected.,0.49
500,Mickelberg,636,Person X won a weightlifting contest at their ...,orig,NaN,4.0,9.0,0.0,5.0,4.0,Competitors are resilient and strong.,Winning required extensive planning and effort.,No harm was caused by winning.,Winning boosts morale and sets an example.,Highly moral as it involves hard work and no h...,Person X,Person X is a dedicated weightlifter.,Other competitors,They are also dedicated weightlifters.,0.41
501,Mickelberg,641,Person X called the bank to tell them about mo...,orig,NaN,3.0,7.0,0.0,7.0,4.0,The bank is resilient and not easily harmed.,Person X acted with deliberate thought.,No harm was caused by the action.,Person X provided substantial assistance by re...,The action was highly moral and responsible.,Person X,Person X is a responsible individual.,The bank,The bank is a financial institution.,2.59


In [45]:
import numpy as np
end_to_end_result["sq_err"] = (end_to_end_result["moral_score"] - end_to_end_result["human_score"]) ** 2
theory_result["mlp_pred"] = np.load("../evaluation/predictions/dyadic_two_layer_mlp.npy")
theory_result["sq_err"] = (theory_result["moral_score"] - theory_result["mlp_pred"]) ** 2


In [46]:
combined_df = pd.DataFrame({
    "human_score": end_to_end_result["human_score"].tolist(),
    "e2e_moral_score": end_to_end_result["moral_score"].tolist(),
    "theory_moral_score": theory_result["moral_score"].tolist(),
    "scenario": theory_result["scenario"].tolist(),
    "id": theory_result["id"].tolist()
})

In [47]:
combined_df["improvement"] = end_to_end_result["sq_err"] - theory_result["sq_err"]

In [52]:
combined_df.sort_values("improvement", ascending=False)["scenario"]

445    You see a coach celebrating with the opposing ...
480    You see a US swimmer cheering as a Chinese foe...
277    You see a teacher ordering a student to get a ...
232    You see the class president saying on TV that ...
269    You see a woman eating dessert before her main...
                             ...                        
141    Person X found a wallet with $50 in it, took t...
142    Person X won the door prize at the town's comm...
143    Person X drove across the country just to see ...
144    You are a lion-tamer on tour with a show of wi...
145    Person X circulated a petition in support of c...
Name: scenario, Length: 503, dtype: object

In [31]:
end_to_end_result

,id,dataset,scenario,circumstance_type,circumstance,moral_score,moral_expl,all_moral_scores,human_score,sq_err
0,147,Mickelberg,Person X successfully remembered their coworke...,orig,NaN,3.0,Remembering coffee orders is neutral and helpful.,[3.0],0.790000,4.884100
1,148,Mickelberg,Person X tried to patch a puncture in the whee...,orig,NaN,0.0,No moral implications in replacing a bike tube.,[0.0],0.340000,0.115600
2,149,Kruepke,I was helping my friend move and she took adva...,orig,NaN,-3.0,Taking advantage of someone's help without gra...,[-3.0],-1.390000,2.592100
3,150,Effron,Provided food and clothing for flood victims,orig,NaN,4.0,Helping flood victims with essentials is highl...,[4],3.226667,0.598044
4,151,Lotto,You are the pilot of a helicopter taking four ...,orig,NaN,-4.0,Sacrificing one life to save others is general...,[-4.0],-1.800000,4.840000
...,...,...,...,...,...,...,...,...,...,...
498,645,Effron,Went to the grocery store for his sick wife,orig,NaN,4.0,Helping a sick spouse is a caring and responsi...,[4],2.675000,1.755625
499,646,Clifford,You see a man in a bar using his phone to watc...,orig,NaN,-4.0,Watching bestiality is highly unethical and il...,[-4.0],-3.300000,0.490000
500,647,Cook,Someone makes a joke about how members of a s...,orig,NaN,-4.0,Disrespectful and promotes harmful stereotypes.,[-4.0],-2.090000,3.648100
501,648,Mickelberg,Person X yelled at an elderly person for walki...,orig,NaN,-4.0,Yelling at an elderly person is disrespectful ...,[-4.0],-2.640000,1.849600
